# Proyek Analisis Sentimen
## Analisis Sentimen Ulasan Aplikasi Shopee Indonesia di Google Play Store

Notebook ini disusun untuk memenuhi kriteria submission proyek analisis sentimen dengan komponen utama:

- data hasil scraping mandiri
- ekstraksi fitur dan pelabelan data
- pelatihan machine learning
- akurasi testing minimal 85%
- inference kelas kategorikal
- tiga percobaan skema pelatihan

Topik yang digunakan:
**Analisis sentimen ulasan aplikasi Shopee Indonesia di Google Play Store**

## 1. Persiapan

Sebelum menjalankan notebook ini, pastikan Anda sudah:

1. Menjalankan file `scraping_google_play.py`
2. Memiliki file dataset `data/shopee_playstore_labeled.csv`
3. Meng-install dependensi dari `requirements.txt`

Jika di Google Colab, jalankan cell instalasi di bawah terlebih dahulu.

In [ ]:
# Jalankan hanya jika di Google Colab / environment baru
# !pip install -r requirements.txt
# Jika requirements.txt belum ada di Colab, install manual:
# !pip install pandas numpy scikit-learn matplotlib seaborn nltk Sastrawi wordcloud emoji transformers datasets accelerate evaluate torch joblib

## 2. Import Library

In [ ]:
import re
import os
import string
import random
import warnings
import joblib
import emoji
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from collections import Counter

import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
sns.set_theme()

nltk.download("stopwords")

## 3. Load Dataset

In [ ]:
DATA_PATH = "data/shopee_playstore_labeled.csv"
TEXT_COLUMN = "content"
LABEL_COLUMN = "label"
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
print("Shape dataset:", df.shape)
df.head()

## 4. Pemeriksaan Awal

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df[LABEL_COLUMN].value_counts()

In [ ]:
plt.figure(figsize=(8, 4))
ax = sns.countplot(data=df, x=LABEL_COLUMN, order=df[LABEL_COLUMN].value_counts().index)
ax.set_title("Distribusi Label")
plt.show()

## 5. Preprocessing Teks

Tahap preprocessing yang digunakan:

- lowercase
- hapus URL
- hapus mention / hashtag
- hapus angka
- hapus emoji
- hapus tanda baca
- normalisasi spasi
- hapus stopword bahasa Indonesia
- stemming menggunakan Sastrawi

Catatan:
Untuk model klasik, preprocessing penuh biasanya membantu.
Untuk IndoBERT, kita akan gunakan teks yang lebih ringan (`text_bert`) agar konteks tetap terjaga.

In [ ]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stopword_id = set(stopwords.words("indonesian"))
custom_stopwords = {
    "yg", "ga", "gak", "nggak", "aja", "nya", "nih", "sih", "dong",
    "udah", "udahh", "banget", "kak", "min", "admin", "aplikasi",
}
stopword_id = stopword_id.union(custom_stopwords)

def clean_text_classic(text: str) -> str:
    text = str(text).lower()
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text)
    text = re.sub(r"#[A-Za-z0-9_]+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()

    tokens = [t for t in text.split() if t not in stopword_id and len(t) > 1]
    stemmed = [stemmer.stem(t) for t in tokens]
    return " ".join(stemmed)

def clean_text_bert(text: str) -> str:
    text = str(text).lower()
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).copy()
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
df = df[df[TEXT_COLUMN] != ""]
df = df.drop_duplicates(subset=[TEXT_COLUMN]).reset_index(drop=True)

df["text_clean"] = df[TEXT_COLUMN].apply(clean_text_classic)
df["text_bert"] = df[TEXT_COLUMN].apply(clean_text_bert)

df = df[df["text_clean"].str.strip() != ""].reset_index(drop=True)

print("Shape dataset setelah preprocessing dasar:", df.shape)
df[[TEXT_COLUMN, "text_clean", "text_bert", LABEL_COLUMN]].head()

## 6. Statistik Ringkas

In [ ]:
df["char_len"] = df[TEXT_COLUMN].astype(str).apply(len)
df["word_len"] = df[TEXT_COLUMN].astype(str).apply(lambda x: len(x.split()))

display(df["char_len"].describe())
display(df["word_len"].describe())

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["word_len"], bins=30)
plt.title("Distribusi Jumlah Kata per Review")
plt.xlabel("Jumlah kata")
plt.ylabel("Frekuensi")
plt.show()

## 7. Split Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text_clean"],
    df[LABEL_COLUMN],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[LABEL_COLUMN]
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))

## 8. Fungsi Evaluasi

In [ ]:
results = []

def evaluate_model(model_name, model, X_train, X_test, y_train, y_test, labels_order=None):
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    print(f"===== {model_name} =====")
    print("Train Accuracy:", round(train_acc, 4))
    print("Test Accuracy :", round(test_acc, 4))
    print("\nClassification Report:")
    print(classification_report(y_test, test_pred))

    cm = confusion_matrix(y_test, test_pred, labels=labels_order)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_order)
    disp.plot()
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

    results.append({
        "model": model_name,
        "train_accuracy": train_acc,
        "test_accuracy": test_acc
    })

    return model

## 9. Eksperimen 1
### TF-IDF + Logistic Regression

In [ ]:
model_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        max_iter=300,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

best_lr = evaluate_model(
    "TF-IDF + Logistic Regression",
    model_lr,
    X_train, X_test, y_train, y_test,
    labels_order=["negative", "neutral", "positive"]
)

## 10. Eksperimen 2
### TF-IDF + LinearSVC

In [ ]:
model_svc = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=40000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("clf", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

best_svc = evaluate_model(
    "TF-IDF + LinearSVC",
    model_svc,
    X_train, X_test, y_train, y_test,
    labels_order=["negative", "neutral", "positive"]
)

## 11. Eksperimen 3
### Deep Learning dengan IndoBERT

Bagian ini menggunakan teks yang diproses ringan (`text_bert`) agar konteks bahasa tetap terjaga.

Jika resource terbatas, eksperimen ini paling cocok dijalankan di Google Colab GPU.

In [ ]:
# Split khusus IndoBERT
X_train_bert, X_test_bert, y_train_bert, y_test_bert = train_test_split(
    df["text_bert"],
    df[LABEL_COLUMN],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[LABEL_COLUMN]
)

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}

train_df_bert = pd.DataFrame({
    "text": X_train_bert.values,
    "label": [label2id[x] for x in y_train_bert]
})

test_df_bert = pd.DataFrame({
    "text": X_test_bert.values,
    "label": [label2id[x] for x in y_test_bert]
})

train_df_bert.head()

In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import evaluate

model_checkpoint = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

train_dataset = Dataset.from_pandas(train_df_bert.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df_bert.reset_index(drop=True))

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = metric_accuracy.compute(predictions=predictions, references=labels)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="weighted")

    return {
        "accuracy": acc["accuracy"],
        "f1_weighted": f1["f1"]
    }

bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="models/indobert_sentiment",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
# Jalankan training IndoBERT
# trainer.train()

In [ ]:
# Evaluasi IndoBERT setelah training
# bert_eval = trainer.evaluate()
# bert_eval

In [ ]:
# Simpan hasil IndoBERT jika sudah selesai training
# trainer.save_model("models/indobert_best")
# tokenizer.save_pretrained("models/indobert_best")

## 12. Perbandingan Hasil Eksperimen

In [ ]:
results_df = pd.DataFrame(results).sort_values(by="test_accuracy", ascending=False)
results_df

## 13. Simpan Model Terbaik Klasik

Bagian ini menyimpan model klasik terbaik ke file `.joblib`.

In [ ]:
# Misal model klasik terbaik adalah LinearSVC
os.makedirs("models", exist_ok=True)
joblib.dump(best_svc, "models/best_classic_sentiment_model.joblib")
print("Model klasik tersimpan di models/best_classic_sentiment_model.joblib")

## 14. Inference / Testing Manual

Cell ini penting karena submission meminta bukti inferensi yang menghasilkan output kelas kategorikal.

In [ ]:
label_examples = [
    "Aplikasinya bagus, pengiriman cepat, saya puas sekali",
    "Biasa saja, tidak terlalu buruk tapi juga tidak istimewa",
    "Aplikasi error terus, checkout gagal, sangat mengecewakan"
]

predictions = best_svc.predict(label_examples)

for text, pred in zip(label_examples, predictions):
    print(f"Teks  : {text}")
    print(f"Pred  : {pred}")
    print("-" * 80)

## 15. Kesimpulan

Isi bagian ini setelah seluruh eksperimen dijalankan. Contoh poin yang bisa Anda tulis:

- Dataset berhasil dikumpulkan melalui scraping mandiri dari Google Play Store
- Pelabelan dilakukan secara otomatis berdasarkan rating bintang
- Tiga eksperimen dilakukan dengan kombinasi metode yang berbeda
- Model terbaik diperoleh dari ...
- Akurasi testing terbaik mencapai ...

Sebaiknya setelah semua selesai, Anda tambahkan interpretasi singkat terhadap confusion matrix dan hasil inference.